# Run VQE warm-started DB-DOI on quantinuum

Following the notebook "vqe_dbqa_synthesis.ipynb" which generates the VQE and GCI circuit, in this notebook, we show how to execute them on Quantinuum emulators and read the results.

## 1. Load VQE and GCI circuits

In [26]:
from pytket import Circuit, OpType
from pytket.circuit.display import render_circuit_jupyter
import pytket.qasm
from datetime import datetime
from quantinuum_utils import *

In [56]:
optimizer = 'cobyla'
nqubits = 10
nlayer = 2
folder_path = f'../results/circuit_qasm/{optimizer}_{nqubits}q_{nlayer}l_XXZ/'
vqe_circ_file = folder_path + 'vqe_circ.qasm'
gci_circ_file = folder_path + 'vqe_gci_circ.qasm'
gci2_circ_file = folder_path + 'vqe_gci_circ_2.qasm'

In [57]:
vqe_circ = pytket.qasm.circuit_from_qasm(vqe_circ_file)
# render_circuit_jupyter(vqe_circ)

In [61]:
gci_circ = pytket.qasm.circuit_from_qasm(gci_circ_file)
gci2_circ = pytket.qasm.circuit_from_qasm(gci2_circ_file)
# render_circuit_jupyter(gci_circ)

Let us check the size of the GCI circuit.

In [62]:
print("--------1 step GCI--------")
print("Circuit depth:", gci_circ.depth())
print("Circuit total gate count:", gci_circ.n_gates)
num_cnots = sum(1 for command in gci_circ if command.op.type == OpType.CX)
print("Circuit CNOT count:", num_cnots)
print("--------2 steps GCI-------")
print("Circuit depth:", gci2_circ.depth())
print("Circuit total gate count:", gci2_circ.n_gates)
num_cnots = sum(1 for command in gci2_circ if command.op.type == OpType.CX)
print("Circuit CNOT count:", num_cnots)

--------1 step GCI--------
Circuit depth: 59
Circuit total gate count: 485
Circuit CNOT count: 0
--------2 steps GCI-------
Circuit depth: 206
Circuit total gate count: 1700
Circuit CNOT count: 0


We also check the theoretical expectance.

In [65]:
import qibo
from qibo import hamiltonians
import matplotlib.pyplot as plt
import numpy as np

def unitary_expectation(H, U=None, ket0=None):
    if ket0 is None:
        ket0 = np.zeros((H.shape[0],), dtype=complex)
        ket0[0] = 1.0
    if U is None:
        psi = ket0
    else:
        psi = U @ ket0
    return np.vdot(psi, H @ psi).real

H = hamiltonians.XXZ(nqubits, 0.5, dense=False)

with open(vqe_circ_file, 'r') as file:
    vqe_qasm_str = file.read()
with open(gci_circ_file, 'r') as file:
    gci_qasm_str = file.read()
with open(gci2_circ_file, 'r') as file:
    gci2_qasm_str = file.read()
vqe_circ_qibo = qibo.Circuit.from_qasm(vqe_qasm_str)
gci_circ_qibo = qibo.Circuit.from_qasm(gci_qasm_str)
gci2_circ_qibo = qibo.Circuit.from_qasm(gci2_qasm_str)
print("VQE:", H.expectation(vqe_circ_qibo().state()))
print("VQE-GCI:", H.expectation(gci_circ_qibo().state()))
print("VQE-GCI 2 steps:", H.expectation(gci2_circ_qibo().state()))

VQE: -14.157823083431683
VQE-GCI: -14.618544270813132
VQE-GCI 2 steps: -14.757341105064263


## 2. Execute on Quantinuum

We only demonstrate the execution of 1 step GCI. For 2 steps, simply change `gci_circ_qibo` to `gci2_circ_qibo` and rename the `qnx_ref` folder path.

### a. Create measurement setup

In [ ]:
# XXZ model (periodic)
nqubits = 10
delta = 0.5
terms = {}
for i in range(nqubits):
    term_x_i = create_qubit_pauli_string(nqubits, {i: Pauli.X, (i+1)%nqubits: Pauli.X}, 1)
    term_y_i = create_qubit_pauli_string(nqubits, {i: Pauli.Y, (i+1)%nqubits: Pauli.Y}, 1)
    term_z_i = create_qubit_pauli_string(nqubits, {i: Pauli.Z, (i+1)%nqubits: Pauli.Z}, delta)
    terms.update(term_x_i)
    terms.update(term_y_i)
    terms.update(term_z_i)
ham_quantinuum = QubitPauliOperator(terms)
terms = [term for term in ham_quantinuum._dict.keys()]
measurement_setup = measurement_reduction(
    terms, strat=PauliPartitionStrat.CommutingSets
)

### b. Upload circuits

In [ ]:
optimisation_level = 2
# create list of circuits for measurement in different bases
vqe_circuit_ref_list = []
gci_circuit_ref_list = []
for i, mc in enumerate(measurement_setup.measurement_circs):
    c = vqe_circ.copy()
    c.append(mc)
    measurement_vqe_circuit_ref = qnx.circuits.upload(
                circuit=c, 
                name=f"measurement vqe circuit {i}",
            )
    # replace with 2 steps GCI circuit
    c = gci_circ.copy()
    c.append(mc)
    measurement_gci_circuit_ref = qnx.circuits.upload(
                circuit=c, 
                name=f"measurement gci circuit {i}",
            )
    vqe_circuit_ref_list.append(measurement_vqe_circuit_ref)
    gci_circuit_ref_list.append(measurement_gci_circuit_ref)

# compile vqe measurement circuit list
compiled_vqe_circuit_refs = qnx.compile(
            name=f"compile_job_VQE_{datetime.now()}",
            circuits=vqe_circuit_ref_list,
            optimisation_level=optimisation_level,
            backend_config=qnx.QuantinuumConfig(device_name="H1-1LE"),
            timeout=None,
        )

# compile gci measurement circuit list
compiled_gci_circuit_refs = qnx.compile(
            name=f"compile_job_GCI_{datetime.now()}",
            circuits=gci_circuit_ref_list,
            optimisation_level=optimisation_level,
            backend_config=qnx.QuantinuumConfig(device_name="H1-1LE"),
            timeout=None,
        )

### c. Execute on emulators

In [ ]:
import os
# Configuration
nshots = 2000
backend_config = qnx.QuantinuumConfig(device_name="H1-1LE")
optimizer = 'cobyla'
# with noise
# backend_config = qnx.QuantinuumConfig(
#     device_name='H1-Emulator',
#     attempt_batching=True,
# )

runs = 6
for _ in range(runs):
    vqe_job_name = f"execute_job_VQE_{nshots}shots_{datetime.now()}"
    gci_job_name = f"execute_job_GCI_{nshots}shots_{datetime.now()}"
    results_vqe = qnx.start_execute_job(
                name=vqe_job_name,
                circuits=compiled_vqe_circuit_refs,
                n_shots=[nshots]*len(vqe_circuit_ref_list),
                backend_config=backend_config,
            )
    results_gci = qnx.start_execute_job(
                name=gci_job_name,
                circuits=compiled_gci_circuit_refs,
                n_shots=[nshots]*len(gci_circuit_ref_list),
                backend_config=backend_config,
            )
    folder_path = f'../results/qnx_job_ref/{optimizer}_{nqubits}q_{nlayer}l_{optimisation_level}ol_XXZ_native_2steps/'
    os.makedirs(folder_path, exist_ok=True)
    qnx.filesystem.save(
        ref=results_vqe,
        path=Path.cwd() / folder_path / vqe_job_name,
        mkdir=True,
    )
    qnx.filesystem.save(
        ref=results_gci,
        path=Path.cwd() /folder_path / gci_job_name,
        mkdir=True,
    )

# Load results from Quantinuum

## 1. 1 Step

In [8]:
optimisation_level = 2
folder_path = Path(f'../results/qnx_job_ref/{optimizer}_{nqubits}q_{nlayer}l_{optimisation_level}ol_XXZ_VQE/')


# Initialize lists
gci_path_list = []
gci_noise_path_list = []
vqe_path_list = []
vqe_noise_path_list = []

for file in folder_path.iterdir():
    if file.is_file():
        name = file.stem  # Use .stem to exclude the file extension
        if name.startswith('execute_job_GCI_') and 'noise' not in name:
            gci_path_list.append(file.resolve())
        elif name.startswith('execute_job_GCI_') and 'noise' in name:
            gci_noise_path_list.append(file.resolve())
        elif name.startswith('execute_job_VQE_') and 'noise' not in name:
            vqe_path_list.append(file.resolve())
        elif name.startswith('execute_job_VQE_') and 'noise' in name:
            vqe_noise_path_list.append(file.resolve())

# verify list length
print('VQE noiseless job counts:', len(vqe_path_list))
print('VQE noise job counts:', len(vqe_noise_path_list))
print('GCI noiseless job counts:', len(gci_path_list))
print('GCI noise job counts:', len(gci_noise_path_list))

VQE noiseless job counts: 12
VQE noise job counts: 8
GCI noiseless job counts: 12
GCI noise job counts: 8


In [ ]:
# load job references
vqe_job_refs = job_ref_from_path_list(vqe_path_list)
vqe_noise_job_refs = job_ref_from_path_list(vqe_noise_path_list)
gci_job_refs = job_ref_from_path_list(gci_path_list)
gci_noise_job_refs = job_ref_from_path_list(gci_noise_path_list)

# load job results
vqe_job_results = load_job_results(vqe_job_refs)
vqe_noise_job_results = load_job_results(vqe_noise_job_refs)
gci_job_results = load_job_results(gci_job_refs)
gci_noise_job_results = load_job_results(gci_noise_job_refs)

In [20]:
vqe_expvals = [compute_expectation_value_from_results(
    job_result, measurement_setup, ham_quantinuum
) for job_result in vqe_job_results]

vqe_noise_expvals = [compute_expectation_value_from_results(
    job_result, measurement_setup, ham_quantinuum
) for job_result in vqe_noise_job_results]

gci_expvals = [compute_expectation_value_from_results(
    job_result, measurement_setup, ham_quantinuum
) for job_result in gci_job_results]

gci_noise_expvals = [compute_expectation_value_from_results(
    job_result, measurement_setup, ham_quantinuum
) for job_result in gci_noise_job_results]

In [ ]:
report_quantinuum = report(vqe_circ_qibo, gci_circ_qibo, H, 2000*2, vqe_expvals, gci_expvals, vqe_noise_expvals, gci_noise_expvals)

In [22]:
report_table(report_quantinuum)

,Analytical,Emulator,Emulator with Noise
VQE energy,-14.1578,-14.1928 ± 0.1626,-14.1402 ± 0.1466
GCI energy,-14.6185,-14.6303 ± 0.0915,-14.5946 ± 0.2328
Difference to target (VQE),1.1183,1.0834 ± 0.1626,1.1359 ± 0.1466
Difference to target (GCI),0.6576,0.6459 ± 0.0915,0.6815 ± 0.2328
Percentage difference to target (VQE),7.32%,7.09% ± 1.06%,7.44% ± 0.96%
Percentage difference to target (GCI),4.30%,4.23% ± 0.60%,4.46% ± 1.52%
Fidelity witness (VQE),-0.0282,0.0039 ± 0.1495,-0.0444 ± 0.1347
Fidelity witness (GCI),0.3954,0.4062 ± 0.1495,0.3734 ± 0.2141


## 2. 2 Steps

In [72]:
optimisation_level = 2
folder_path = Path(f'../results/qnx_job_ref/{optimizer}_{nqubits}q_{nlayer}l_{optimisation_level}ol_XXZ_VQE_2steps/')


# Initialize lists
gci_path_list = []
gci_noise_path_list = []
vqe_path_list = []
vqe_noise_path_list = []

for file in folder_path.iterdir():
    if file.is_file():
        name = file.stem  # Use .stem to exclude the file extension
        if name.startswith('execute_job_GCI_') and 'noise' not in name:
            gci_path_list.append(file.resolve())
        elif name.startswith('execute_job_GCI_') and 'noise' in name:
            gci_noise_path_list.append(file.resolve())
        elif name.startswith('execute_job_VQE_') and 'noise' not in name:
            vqe_path_list.append(file.resolve())
        elif name.startswith('execute_job_VQE_') and 'noise' in name:
            vqe_noise_path_list.append(file.resolve())

# verify list length
print('VQE noiseless job counts:', len(vqe_path_list))
print('VQE noise job counts:', len(vqe_noise_path_list))
print('GCI noiseless job counts:', len(gci_path_list))
print('GCI noise job counts:', len(gci_noise_path_list))

VQE noiseless job counts: 6
VQE noise job counts: 6
GCI noiseless job counts: 6
GCI noise job counts: 6


In [73]:
# load job references
vqe_job_refs = job_ref_from_path_list(vqe_path_list)
vqe_noise_job_refs = job_ref_from_path_list(vqe_noise_path_list)
gci_job_refs = job_ref_from_path_list(gci_path_list)
gci_noise_job_refs = job_ref_from_path_list(gci_noise_path_list)

# load job results
vqe_job_results = load_job_results(vqe_job_refs)
vqe_noise_job_results = load_job_results(vqe_noise_job_refs)
gci_job_results = load_job_results(gci_job_refs)
gci_noise_job_results = load_job_results(gci_noise_job_refs)

Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider 

In [74]:
vqe_expvals = [compute_expectation_value_from_results(
    job_result, measurement_setup, ham_quantinuum
) for job_result in vqe_job_results]

vqe_noise_expvals = [compute_expectation_value_from_results(
    job_result, measurement_setup, ham_quantinuum
) for job_result in vqe_noise_job_results]

gci_expvals = [compute_expectation_value_from_results(
    job_result, measurement_setup, ham_quantinuum
) for job_result in gci_job_results]

gci_noise_expvals = [compute_expectation_value_from_results(
    job_result, measurement_setup, ham_quantinuum
) for job_result in gci_noise_job_results]

In [75]:
report_quantinuum2 = report(vqe_circ_qibo, gci2_circ_qibo, H, 2000*2, vqe_expvals, gci_expvals, vqe_noise_expvals, gci_noise_expvals)
report_table(report_quantinuum2)

,Analytical,Emulator,Emulator with Noise
VQE energy,-14.1578,-14.1985 ± 0.0684,-13.9279 ± 0.1957
GCI energy,-14.7573,-14.7481 ± 0.1437,-13.6482 ± 0.1217
Difference to target (VQE),1.1183,1.0776 ± 0.0684,1.3482 ± 0.1957
Difference to target (GCI),0.5188,0.5280 ± 0.1437,1.6280 ± 0.1217
Percentage difference to target (VQE),7.32%,7.05% ± 0.45%,8.83% ± 1.28%
Percentage difference to target (GCI),3.40%,3.46% ± 0.94%,10.66% ± 0.80%
Fidelity witness (VQE),-0.0282,0.0092 ± 0.0629,-0.2395 ± 0.1800
Fidelity witness (GCI),0.5230,0.5145 ± 0.0629,-0.4968 ± 0.1119
